In [1]:
import pandas as pd

df = pd.read_csv("lifecyclesavings.csv")
print(df.shape)
print(df.isna().sum())
print(df.mean(numeric_only=True))


(50, 6)
Contry    0
sr        0
pop15     0
pop75     0
dpi       0
ddpi      0
dtype: int64
sr          9.6710
pop15      35.0896
pop75       2.2930
dpi      1106.7584
ddpi        3.7576
dtype: float64


In [4]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
 
df = pd.read_csv("lifecyclesavings.csv")
df = df.rename(columns={"Contry": "country"})
features = ["sr", "pop15", "pop75", "dpi", "ddpi"]
 
# Standardize: dpi is in the thousands, pop75 is single digits -> unscaled, dpi would dominate distances
X = StandardScaler().fit_transform(df[features])
 
# Pick k by silhouette
print("k  silhouette")
for k in range(2, 7):
    km = KMeans(n_clusters=k, n_init=20, random_state=42).fit(X)
    print(k, round(silhouette_score(X, km.labels_), 3))
 
K = 3
km = KMeans(n_clusters=K, n_init=20, random_state=42).fit(X)
df["cluster"] = km.labels_
 
print("\nCluster sizes:\n", df["cluster"].value_counts().sort_index())
print("\nCluster means:\n", df.groupby("cluster")[features].mean().round(2))
print("\nOverall means:\n", df[features].mean().round(2))
for c in sorted(df["cluster"].unique()):
    print(f"\nCluster {c}:", ", ".join(df.loc[df.cluster == c, "country"]))
 
# 2-D view via PCA
pca = PCA(n_components=2)
P = pca.fit_transform(X)
print("\nPCA variance explained:", pca.explained_variance_ratio_.round(3))
 
fig, ax = plt.subplots(figsize=(11, 8))
colors = ["#1f77b4", "#d95f02", "#1b9e77", "#7570b3", "#e7298a"]
for c in range(K):
    m = df.cluster == c
    ax.scatter(P[m, 0], P[m, 1], s=70, color=colors[c], label=f"Cluster {c} (n={m.sum()})",
               edgecolor="white", linewidth=0.8)
for i, name in enumerate(df["country"]):
    ax.annotate(name, (P[i, 0], P[i, 1]), fontsize=7, xytext=(4, 3), textcoords="offset points")
ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]:.0%} of variance)")
ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]:.0%} of variance)")
ax.set_title("Countries clustered by savings behavior (K-means, k=3, standardized features)")
ax.legend()
ax.grid(alpha=0.25)
fig.tight_layout()
fig.savefig("savings_clusters.png", dpi=150)
 
df.to_csv("savings_with_clusters.csv", index=False)


k  silhouette
2 0.436
3 0.422
4 0.366
5 0.311
6 0.321

Cluster sizes:
 cluster
0     7
1    22
2    21
Name: count, dtype: int64

Cluster means:
             sr  pop15  pop75      dpi  ddpi
cluster                                    
0        13.73  37.62   1.75   481.39  8.91
1        10.89  26.50   3.56  2012.22  3.38
2         7.04  43.24   1.15   366.63  2.44

Overall means:
 sr          9.67
pop15      35.09
pop75       2.29
dpi      1106.76
ddpi        3.76
dtype: float64

Cluster 0: China, Japan, Malta, Portugal, Zambia, Jamaica, Libya

Cluster 1: Australia, Austria, Belgium, Canada, Denmark, Finland, France, Germany, Greece, Iceland, Ireland, Italy, Luxembourg, Norway, Netherlands, New Zealand, Spain, Sweden, Switzerland, United Kingdom, United States, Uruguay

Cluster 2: Bolivia, Brazil, Chile, Colombia, Costa Rica, Ecuador, Guatamala, Honduras, India, Korea, Nicaragua, Panama, Paraguay, Peru, Philippines, South Africa, South Rhodesia, Turkey, Tunisia, Venezuela, Malaysia

PCA

C:\ProgramData\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1428: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(
C:\ProgramData\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1428: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(
C:\ProgramData\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1428: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(
C:\ProgramData\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1428: UserWarning: KMeans is known to have a memory leak on Window

In [7]:
df = pd.read_csv("savings_with_clusters.csv")
cols = ["sr", "pop15", "pop75", "dpi", "ddpi"]

group_means = df.groupby("cluster")[cols].mean().round(2)

print(group_means.loc[1])
print(df[cols].mean().round(2))

sr         10.89
pop15      26.50
pop75       3.56
dpi      2012.22
ddpi        3.38
Name: 1, dtype: float64
sr          9.67
pop15      35.09
pop75       2.29
dpi      1106.76
ddpi        3.76
dtype: float64


In [ ]:
#It is different than what the AI shows. Group 1 has a higher dpi, higher pop75, and lower pop15.
#The caution with trusting AI analysis is that you do not know what it is doing unless you know the techniques.
#As well as be able to read what it is doing.